# Checkpoint 1 — Inflação e Taxa Selic no Brasil
Análise exploratória com dados reais do Banco Central do Brasil (SGS). Recorte: 2020 até 30/08/2026.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

import requests

BASE_URL = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
DATA_INICIAL = "01/01/2020"
DATA_FINAL = "30/08/2026"

def buscar_serie(codigo, data_inicial=DATA_INICIAL, data_final=DATA_FINAL):
    params = {
        "formato": "json",
        "dataInicial": data_inicial,
        "dataFinal": data_final,
    }
    resposta = requests.get(BASE_URL.format(codigo=codigo), params=params, timeout=30)
    resposta.raise_for_status()
    dados = pd.DataFrame(resposta.json())
    dados["data"] = pd.to_datetime(dados["data"], format="%d/%m/%Y")
    dados["valor"] = pd.to_numeric(dados["valor"], errors="coerce")
    return dados.dropna().drop_duplicates().sort_values("data")

def preparar_base():
    ipca = buscar_serie(433).rename(columns={"valor": "ipca_mensal"})
    selic = buscar_serie(1178).rename(columns={"valor": "selic_anual"})

    ipca["ano_mes"] = ipca["data"].dt.to_period("M")
    selic["ano_mes"] = selic["data"].dt.to_period("M")

    selic_mensal = (
        selic.groupby("ano_mes", as_index=False)["selic_anual"]
        .mean()
        .rename(columns={"selic_anual": "selic_media_mensal"})
    )

    base = ipca[["data", "ano_mes", "ipca_mensal"]].merge(
        selic_mensal, on="ano_mes", how="inner"
    )

    base["ipca_12m"] = (
        (1 + base["ipca_mensal"] / 100)
        .rolling(12)
        .apply(lambda valores: valores.prod(), raw=True)
        .sub(1)
        .mul(100)
    )

    return ipca, selic, base

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()


## 1. Coleta e integração
IPCA (SGS 433) é mensal. Selic (SGS 1178) é diária e será agregada pela média mensal antes da integração.

In [ ]:
ipca, selic, base = preparar_base()
print('IPCA:', ipca.shape)
print('Selic:', selic.shape)
print('Base integrada:', base.shape)
base.head()

## 2. Qualidade e descrição dos dados

In [ ]:
print('Valores ausentes:')
display(base.isna().sum())
print('Duplicados:', base.duplicated().sum())
display(base[['ipca_mensal','selic_media_mensal','ipca_12m']].describe().round(2))

## 3. Pergunta 1 — Como a inflação evoluiu desde 2020?
**KPI:** IPCA acumulado em 12 meses.

In [ ]:
validos = base.dropna(subset=['ipca_12m']).copy()
pico = validos.loc[validos['ipca_12m'].idxmax()]
minimo = validos.loc[validos['ipca_12m'].idxmin()]
atual = validos.iloc[-1]
print(f"Maior IPCA 12m: {pico['ipca_12m']:.2f}% em {pico['ano_mes']}")
print(f"Menor IPCA 12m: {minimo['ipca_12m']:.2f}% em {minimo['ano_mes']}")
print(f"Último IPCA 12m disponível: {atual['ipca_12m']:.2f}% em {atual['ano_mes']}")

plt.figure(figsize=(12,5))
plt.plot(validos['data'], validos['ipca_12m'])
plt.axhline(4.5, linestyle='--', label='Referência de 4,5%')
plt.title('IPCA acumulado em 12 meses')
plt.ylabel('%')
plt.grid(alpha=.25)
plt.legend()
plt.show()

## 4. Pergunta 2 — Como a Selic reagiu aos períodos de maior inflação?
**KPI:** Selic média mensal comparada ao IPCA acumulado em 12 meses.

In [ ]:
fig, ax1 = plt.subplots(figsize=(12,5))
ax1.plot(validos['data'], validos['ipca_12m'], label='IPCA 12m')
ax1.set_ylabel('IPCA 12m (%)')
ax2 = ax1.twinx()
ax2.plot(validos['data'], validos['selic_media_mensal'], linestyle='--', label='Selic média mensal')
ax2.set_ylabel('Selic (% a.a.)')
ax1.set_title('Inflação acumulada e taxa Selic')
ax1.grid(alpha=.25)
plt.show()

maior_selic = validos.loc[validos['selic_media_mensal'].idxmax()]
print(f"Maior Selic média mensal: {maior_selic['selic_media_mensal']:.2f}% a.a. em {maior_selic['ano_mes']}")

## 5. Pergunta 3 — Qual a intensidade da relação entre inflação e juros?
**KPI:** correlação de Pearson. Correlação mede associação linear, não causalidade.

In [ ]:
correlacao = validos['ipca_12m'].corr(validos['selic_media_mensal'])
print(f'Correlação IPCA 12m x Selic: {correlacao:.3f}')

plt.figure(figsize=(8,5))
plt.scatter(validos['ipca_12m'], validos['selic_media_mensal'], alpha=.7)
plt.xlabel('IPCA acumulado em 12 meses (%)')
plt.ylabel('Selic média mensal (% a.a.)')
plt.title('Relação entre inflação e juros')
plt.grid(alpha=.25)
plt.show()

## 6. Indicadores e apoio à decisão
O **IPCA em 12 meses** resume a pressão inflacionária; a **Selic média mensal** representa a postura dos juros; e a **correlação** resume a associação linear entre os ciclos. Em conjunto, esses indicadores ajudam a avaliar poder de compra, custo do crédito e ambiente para consumo e investimento.

A análise deve considerar defasagens: decisões de juros podem levar meses para afetar a inflação e também respondem a expectativas e outras condições macroeconômicas.

In [ ]:
saida = ROOT / 'data' / 'economia_ipca_selic.csv'
saida.parent.mkdir(exist_ok=True)
base.to_csv(saida, index=False)
print(f'Base integrada salva em {saida}')